# SESSION 2.2: NEURAL NETWORKS - THEORY

---

## **WHAT IS A NEURAL NETWORK?**

### **The Big Idea**

A neural network is a **function approximator**. It learns to map inputs to outputs by finding the right parameters (weights and biases).

```
Input → [Neural Network] → Output
Audio → [Magic Box]      → "2 speakers"
```

Your job: Train the network so the "magic box" learns the right mapping.

---

## **THE SIMPLEST BUILDING BLOCK: A NEURON**

### **What Does One Neuron Do?**

A neuron performs two operations:
1. **Weighted sum** of inputs
2. **Activation** function

```
Inputs: [x₁, x₂, x₃]
Weights: [w₁, w₂, w₃]
Bias: b

Step 1: Weighted sum
z = (x₁ × w₁) + (x₂ × w₂) + (x₃ × w₃) + b

Step 2: Activation
output = activation(z)
```

### **Visual Example**

```
Input features:
x₁ = 0.5  (average amplitude)
x₂ = 0.3  (energy in low frequencies)
x₃ = 0.8  (energy in high frequencies)

Weights (learned during training):
w₁ = 2.0
w₂ = -1.0
w₃ = 1.5

Bias:
b = 0.1

Computation:
z = (0.5 × 2.0) + (0.3 × -1.0) + (0.8 × 1.5) + 0.1
z = 1.0 - 0.3 + 1.2 + 0.1
z = 2.0

After ReLU activation:
output = max(0, 2.0) = 2.0
```

---

## **WHY ACTIVATION FUNCTIONS?**

### **The Problem Without Activation**

Without activation functions, stacking multiple layers is useless:

```
Layer 1: y = W₁x + b₁
Layer 2: z = W₂y + b₂
        = W₂(W₁x + b₁) + b₂
        = (W₂W₁)x + (W₂b₁ + b₂)
        = W_combined × x + b_combined
```

**This is just one linear transformation!** No matter how many layers, it collapses to a single linear function.

### **The Solution: Non-linearity**

Activation functions introduce **non-linearity**, allowing networks to learn complex patterns:

```
Layer 1: y = ReLU(W₁x + b₁)
Layer 2: z = ReLU(W₂y + b₂)
```

Now this **cannot** be simplified to a single linear function! The network can learn curves, boundaries, and complex decision surfaces.

---

## **COMMON ACTIVATION FUNCTIONS**

### **1. ReLU (Rectified Linear Unit)**
```
ReLU(x) = max(0, x)

Input:  [-2, -1, 0, 1, 2]
Output: [ 0,  0, 0, 1, 2]
```

**Why it's popular:**
- ✅ Simple and fast
- ✅ Helps avoid vanishing gradients
- ✅ Works well in practice

**Graph:**
```
      |
    2 |         /
    1 |        /
    0 |_______/
      |
   -2 -1 0 1 2
```

### **2. Sigmoid**
```
σ(x) = 1 / (1 + e^(-x))

Input:  [-2, -1, 0, 1, 2]
Output: [0.12, 0.27, 0.5, 0.73, 0.88]
```

**Use case:** Binary classification (outputs probability 0-1)

**Graph:**
```
    1 |     ___
      |    /
  0.5 |   /
      |  /
    0 |_/___
     -2  0  2
```

### **3. Tanh (Hyperbolic Tangent)**
```
tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))

Input:  [-2, -1, 0, 1, 2]
Output: [-0.96, -0.76, 0, 0.76, 0.96]
```

**Use case:** When you want outputs in range [-1, 1]

---

## **FROM NEURON TO LAYER**

### **A Layer = Many Neurons in Parallel**

Instead of one neuron, use many:

```
Input: [x₁, x₂, x₃]

Neuron 1: output₁ = ReLU(w₁₁x₁ + w₁₂x₂ + w₁₃x₃ + b₁)
Neuron 2: output₂ = ReLU(w₂₁x₁ + w₂₂x₂ + w₂₃x₃ + b₂)
Neuron 3: output₃ = ReLU(w₃₁x₁ + w₃₂x₂ + w₃₃x₃ + b₃)
...
Neuron N: outputₙ = ReLU(wₙ₁x₁ + wₙ₂x₂ + wₙ₃x₃ + bₙ)

Layer output: [output₁, output₂, output₃, ..., outputₙ]
```

### **Matrix Form (Much Faster!)**

Instead of computing each neuron separately, use matrix multiplication:

```
Input shape: (3,)         [x₁, x₂, x₃]
Weight matrix: (N, 3)     N neurons, each with 3 weights
Bias vector: (N,)         One bias per neuron

Output = ReLU(W @ x + b)
```

**Example with 4 neurons:**
```python
x = [0.5, 0.3, 0.8]  # 3 inputs

W = [[2.0, -1.0, 1.5],   # Neuron 1's weights
     [0.5,  1.0, 0.2],   # Neuron 2's weights
     [1.0,  0.0, -1.0],  # Neuron 3's weights
     [-0.5, 2.0, 0.3]]   # Neuron 4's weights

b = [0.1, -0.2, 0.5, 0.0]

z = W @ x + b = [2.0, 0.79, -0.3, 0.19]
output = ReLU(z) = [2.0, 0.79, 0.0, 0.19]
```

---

## **MULTI-LAYER NETWORK (THE FULL PICTURE)**

### **Architecture**

```
Input Layer → Hidden Layer 1 → Hidden Layer 2 → Output Layer
```

**Concrete example for speaker counting:**

```
Audio features (10,)
      ↓
   [Layer 1: 10 → 64]  ← 64 neurons
      ↓ ReLU
   [Layer 2: 64 → 32]  ← 32 neurons
      ↓ ReLU
   [Layer 3: 32 → 3]   ← 3 neurons (1, 2, or 3 speakers)
      ↓ Softmax
   Probabilities [0.1, 0.7, 0.2]
      ↓
   Prediction: 2 speakers (highest probability)
```

### **Information Flow (Forward Pass)**

```python
# Input
x = audio_features  # Shape: (10,)

# Layer 1
z1 = W1 @ x + b1           # Shape: (64,)
h1 = ReLU(z1)              # Shape: (64,)

# Layer 2
z2 = W2 @ h1 + b2          # Shape: (32,)
h2 = ReLU(z2)              # Shape: (32,)

# Layer 3 (output)
z3 = W3 @ h2 + b3          # Shape: (3,)
output = softmax(z3)       # Shape: (3,) - probabilities

# Prediction
predicted_class = argmax(output)  # 0, 1, or 2
```

---

## **LOSS FUNCTIONS (HOW TO MEASURE ERROR)**

### **The Problem**

The network makes predictions, but how do we know if they're good?

```
True label: 2 speakers (class index 1)
Prediction: [0.1, 0.7, 0.2]  ← 70% confident it's 2 speakers

Is this good? How do we quantify "goodness"?
```

### **Loss = Measure of Error**

Loss is a single number that tells us how wrong the prediction is.

**Goal:** Minimize loss → Better predictions

---

### **1. Cross-Entropy Loss (Classification)**

Used when predicting classes (like speaker count: 1, 2, or 3).

**Formula:**
```
Loss = -log(predicted_probability_of_correct_class)
```

**Example:**

```python
True label: class 1 (2 speakers)
Prediction: [0.1, 0.7, 0.2]

Loss = -log(0.7) = 0.357
```

**Intuition:**
- Prediction close to 1.0 → Loss close to 0 (good!)
- Prediction close to 0.0 → Loss goes to infinity (bad!)

**Why negative log?**
- log(1.0) = 0 → Perfect prediction has zero loss
- log(0.5) = -0.69 → Uncertain prediction has higher loss
- log(0.1) = -2.3 → Wrong prediction has very high loss
- Negative makes it positive (we want to minimize positive numbers)

### **2. Mean Squared Error (Regression)**

Used when predicting continuous values.

**Formula:**
```
MSE = (1/N) × Σ(predicted - true)²
```

**Example:**
```python
True value: 5.0
Prediction: 4.5

Error = 5.0 - 4.5 = 0.5
Squared error = 0.5² = 0.25
```

---

## **BACKPROPAGATION (THE LEARNING ALGORITHM)**

### **The Big Picture**

1. **Forward pass:** Input → Network → Prediction
2. **Compute loss:** Compare prediction to true label
3. **Backward pass:** Compute gradients (how to adjust weights)
4. **Update weights:** Move in direction that reduces loss

### **Gradient = Direction to Improve**

Imagine you're on a hill and want to go down (minimize loss):

```
      🏔️
     /  \
    /    \
   /  👤  \    ← You are here
  /        \
 /          \
/____________\
   Minimum (goal)
```

**Gradient tells you:** "Go left to go downhill"

In neural networks:
- Gradient tells you how to adjust each weight
- Positive gradient → Decrease weight
- Negative gradient → Increase weight

### **Chain Rule (How Backprop Works)**

The loss depends on the output, which depends on layer 2, which depends on layer 1, which depends on weights.

```
Loss
  ↑
Output Layer
  ↑
Hidden Layer 2
  ↑
Hidden Layer 1
  ↑
Weights
```

**Chain rule:** To find how loss changes with respect to early weights, multiply gradients backwards through the chain.

**PyTorch does this automatically with `.backward()`!**

---

## **TRAINING LOOP (PUTTING IT ALL TOGETHER)**

```python
for epoch in range(num_epochs):
    for batch in dataloader:
        # 1. Get data
        audio, labels = batch
        
        # 2. Forward pass
        predictions = model(audio)
        
        # 3. Compute loss
        loss = loss_function(predictions, labels)
        
        # 4. Backward pass (compute gradients)
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        # 6. Zero gradients for next iteration
        optimizer.zero_grad()
```

### **Step-by-Step Breakdown**

**Step 1: Get data**
```python
audio shape: (32, 1, 48000)   # 32 samples in batch
labels shape: (32,)            # 32 labels (0, 1, or 2)
```

**Step 2: Forward pass**
```python
predictions = model(audio)
# predictions shape: (32, 3)  # 3 class probabilities per sample
```

**Step 3: Compute loss**
```python
loss = CrossEntropyLoss(predictions, labels)
# loss: single number (e.g., 1.234)
```

**Step 4: Backward pass**
```python
loss.backward()
# Computes gradients for ALL parameters
# Stores them in param.grad
```

**Step 5: Update weights**
```python
optimizer.step()
# For each weight: w = w - learning_rate × gradient
```

**Step 6: Zero gradients**
```python
optimizer.zero_grad()
# Clear gradients (they accumulate otherwise!)
```

---

## **HYPERPARAMETERS**

### **What Are They?**

Settings you choose **before** training that control the learning process.

### **Key Hyperparameters**

**1. Learning Rate**
```
How big are the weight updates?

Too large: Jumps around, never converges
Too small: Learns too slowly, gets stuck

Typical values: 0.001, 0.0001
```

**2. Batch Size**
```
How many samples to process before updating weights?

Larger batch: More stable gradients, but uses more memory
Smaller batch: More updates, but noisier gradients

Typical values: 16, 32, 64
```

**3. Number of Epochs**
```
How many times to see the entire dataset?

Too few: Underfitting (hasn't learned enough)
Too many: Overfitting (memorized training data)

Typical values: 10-100
```

**4. Hidden Layer Sizes**
```
How many neurons in each layer?

More neurons: More capacity, but slower and can overfit
Fewer neurons: Faster, but might not learn complex patterns

Typical values: 64, 128, 256
```

---

## **OVERFITTING VS UNDERFITTING**

### **Underfitting**
```
Training loss: High
Validation loss: High

Problem: Model is too simple, hasn't learned the patterns

Solution:
- Bigger network (more layers/neurons)
- Train longer
- Better features
```

### **Overfitting**
```
Training loss: Very low
Validation loss: High

Problem: Model memorized training data, doesn't generalize

Solution:
- More training data
- Regularization (dropout, weight decay)
- Simpler model
- Early stopping
```

### **Just Right**
```
Training loss: Low
Validation loss: Low (similar to training)

✅ Model has learned generalizable patterns!
```

**Visual:**
```
Underfitting:        Just Right:        Overfitting:
    ●  ●                 ●──●                ●──●
  ●      ●             ●      ●            ●│    │●
●          ●         ●          ●        ●─┘    └─●
  ●      ●             ●      ●            ●      ●
    ●  ●                 ●──●                ●──●
    
Simple line      Captures pattern    Memorizes noise
```

---

## **AUDIO-SPECIFIC CONSIDERATIONS**

### **Why Neural Networks for Audio?**

Traditional approaches:
- Hand-crafted features (MFCCs, spectrograms)
- Manual feature engineering
- Limited by human intuition

Neural networks:
- Learn features automatically
- Can discover patterns humans miss
- End-to-end learning (raw audio → prediction)

### **Common Audio Features (Before Deep Learning)**

```
Raw Audio (48000 samples)
         ↓
  Extract Features
         ↓
[Energy, Zero-crossing rate, Spectral centroid, MFCCs, ...]
         ↓
   Traditional ML (SVM, Random Forest)
         ↓
    Prediction
```

### **Deep Learning Approach**

```
Raw Audio (48000 samples)
         ↓
   Neural Network
         ↓
    Prediction

(Features learned automatically!)
```

---

## **KEY CONCEPTS SUMMARY**

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Neuron** | Weighted sum + activation | Building block of network |
| **Layer** | Many neurons in parallel | Learns multiple features |
| **Activation** | Non-linear function | Enables complex patterns |
| **Forward Pass** | Input → Output | Makes predictions |
| **Loss** | Error measurement | Quantifies how wrong we are |
| **Backprop** | Gradient computation | Tells us how to improve |
| **Optimizer** | Weight update rule | Actually improves the network |

---

In [ ]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
import os
from typing import List, Tuple, Dict
import sys

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio

# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

/Users/apple/Documents/programing/projects/voice_isolation
✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


## **TODO 1: BUILD A SINGLE NEURON**

**Goal:** Create one neuron that computes weighted sum + activation

**Requirements:**
- Input: 1D tensor (features)
- Weights: 1D tensor (same size as input)
- Bias: scalar
- Output: single number after ReLU

**Your task:** Write a function `neuron(x, w, b)` that returns `ReLU(w·x + b)`

**Hints:**
- Use `torch.dot()` or `@` for dot product
- ReLU = `torch.clamp(value, min=0)` or `torch.max(value, torch.tensor(0.0))`
- Or just use `torch.nn.functional.relu()`

**Test case:**
```python
x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([2.0, -1.0, 1.5])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")
# Expected: 2.0 (from theory example)
```

**After it works - EXPERIMENT:**
- What happens if all weights are zero?
- What if input has negative values?
- Try different bias values - how does it shift the output?
- Replace ReLU with sigmoid - how do outputs differ?

---

In [3]:
def neuron(
        x: torch.Tensor,
        w: torch.Tensor,
        b: float
) -> float:
    z1 = w @ x + b
    
    return torch.relu(z1)

x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([2.0, -1.0, 1.5])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")

Output: 2.0


In [4]:
x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([0.1, 0.0, 0.0])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")

Output: 0.15000000596046448


In [5]:
x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([0.0, 0.0, 0.0])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")

Output: 0.10000000149011612


In [6]:
x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([1.0, 1.0, 1.0])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")

Output: 1.6999999284744263


In [7]:
x = torch.tensor([0.5, 0.3, 0.8])
w = torch.tensor([-1.0, -4.2, 1.0])
b = torch.tensor(0.1)

output = neuron(x, w, b)
print(f"Output: {output}")

Output: 0.0


## **TODO 2: BUILD A LAYER (MULTIPLE NEURONS)**

**Goal:** Create a layer with N neurons computing in parallel

**Requirements:**
- Input: (input_size,) tensor
- Weights: (num_neurons, input_size) matrix
- Bias: (num_neurons,) vector
- Output: (num_neurons,) tensor after ReLU

**Your task:** Write a function `layer(x, W, b)` that returns `ReLU(W @ x + b)`

**Hints:**
- Matrix multiplication: `W @ x`
- Broadcasting handles bias addition automatically
- Apply ReLU to the entire result

**Test case:**
```python
x = torch.tensor([0.5, 0.3, 0.8])  # 3 inputs
W = torch.randn(5, 3)  # 5 neurons, each with 3 weights
b = torch.randn(5)      # 5 biases

output = layer(x, W, b)
print(f"Output shape: {output.shape}")  # Should be (5,)
print(f"All non-negative? {(output >= 0).all()}")  # Should be True (ReLU)
```

**After it works - EXPERIMENT:**
- Increase number of neurons to 100 - does it get slower?
- Print the weight matrix - what do the values look like?
- What percentage of outputs are exactly zero after ReLU?
- Try without ReLU - how do outputs change?

---

In [8]:
def layer(
        x: torch.Tensor,
        W: torch.Tensor,
        b: torch.Tensor
) -> torch.Tensor:
    z1 = W @ x + b
    
    return torch.relu(z1)


x = torch.tensor([0.5, 0.3, 0.8])  # 3 inputs
W = torch.randn(5, 3)  # 5 neurons, each with 3 weights
b = torch.randn(5)      # 5 biases

output = layer(x, W, b)
print(f"Output shape: {output.shape}")  # Should be (5,)
print(f"All non-negative? {(output >= 0).all()}")  # Should be True (ReLU)

Output shape: torch.Size([5])
All non-negative? True


In [9]:
x = torch.randn(100)  # 3 inputs
W = torch.randn(500000, 100)  # 5 neurons, each with 3 weights
b = torch.randn(500000)      # 5 biases

output = layer(x, W, b)
print(f"Output shape: {output.shape}")  # Should be (5,)
print(f"All non-negative? {(output >= 0).all()}")  # Should be True (ReLU)

Output shape: torch.Size([500000])
All non-negative? True


## **TODO 3: BUILD A 2-LAYER NETWORK**

**Goal:** Stack two layers together

**Requirements:**
- Input: (input_size,)
- Hidden layer: (hidden_size, input_size)
- Output layer: (output_size, hidden_size)
- Connect them: input → hidden (ReLU) → output

**Your task:** Write a function `two_layer_network(x, W1, b1, W2, b2)` that:
1. Computes hidden = ReLU(W1 @ x + b1)
2. Computes output = W2 @ hidden + b2
3. Returns output

**Note:** No ReLU on final layer! (We'll add softmax later for classification)

**Test case:**
```python
x = torch.randn(10)      # 10 input features
W1 = torch.randn(64, 10)  # Hidden layer: 64 neurons
b1 = torch.randn(64)
W2 = torch.randn(3, 64)   # Output layer: 3 classes
b2 = torch.randn(3)

output = two_layer_network(x, W1, b1, W2, b2)
print(f"Output shape: {output.shape}")  # Should be (3,)
print(f"Output: {output}")
```

**After it works - EXPERIMENT:**
- Add a 3rd layer - where does it go in the computation?
- Make hidden layer very small (5 neurons) - what happens to output?
- Make it very large (1000 neurons) - does output quality change?
- Print intermediate hidden layer values - what do they look like?

---

In [10]:
def two_layer_network(
        x: torch.Tensor,
        W1: torch.Tensor,
        b1: torch.Tensor,
        W2: torch.Tensor,
        b2: torch.Tensor
) -> torch.Tensor:
    z1 = W1 @ x + b1
    h1 = torch.relu(z1)  
    
    output = W2 @ h1 + b2  
    
    return output

x = torch.randn(10)      # 10 input features
W1 = torch.randn(64, 10)  # Hidden layer: 64 neurons
b1 = torch.randn(64)
W2 = torch.randn(3, 64)   # Output layer: 3 classes
b2 = torch.randn(3)

output = two_layer_network(x, W1, b1, W2, b2)
print(f"Output shape: {output.shape}")  # Should be (3,)
print(f"Output: {output}")

Output shape: torch.Size([3])
Output: tensor([18.0674, 44.7524,  9.5616])


In [11]:
x = torch.randn(10)      # 10 input features
W1 = torch.randn(5, 10)  # Hidden layer: 64 neurons
b1 = torch.randn(5)
W2 = torch.randn(3, 5)   # Output layer: 3 classes
b2 = torch.randn(3)

output = two_layer_network(x, W1, b1, W2, b2)
print(f"Output shape: {output.shape}")  # Should be (3,)
print(f"Output: {output}")

Output shape: torch.Size([3])
Output: tensor([ 0.3799,  0.4558, -0.3396])


In [12]:
def three_layer_network(
        x: torch.Tensor,
        W1: torch.Tensor,
        b1: torch.Tensor,
        W2: torch.Tensor,
        b2: torch.Tensor,
        W3: torch.Tensor,
        b3: torch.Tensor
) -> torch.Tensor:
    z1 = W1 @ x + b1
    h1 = torch.relu(z1)
    
    z2 = W2 @ h1 + b2
    h2 = torch.relu(z2)
    
    output = W3 @ h2 + b3
    
    return output

x = torch.randn(10)      # 10 input features
W1 = torch.randn(128, 10)  # Hidden layer: 64 neurons
b1 = torch.randn(128)
W2 = torch.randn(64, 128)  # Hidden layer: 64 neurons
b2 = torch.randn(64)
W3 = torch.randn(3, 64)   # Output layer: 3 classes
b3 = torch.randn(3)

output = three_layer_network(x, W1, b1, W2, b2, W3, b3)
print(f"Output shape: {output.shape}")  # Should be (3,)
print(f"Output: {output}")

Output shape: torch.Size([3])
Output: tensor([-50.2988,   4.3285, -74.8040])


## **TODO 4: COMPUTE LOSS (CROSS-ENTROPY)**

**Goal:** Measure how wrong your prediction is

**Requirements:**
- Input: Network output (3,) - raw scores (logits)
- Target: True class index (0, 1, or 2)
- Apply softmax to convert logits to probabilities
- Compute negative log probability of correct class

**Your task:** Write a function `cross_entropy_loss(logits, target)`

**Hints:**
- Softmax: `torch.nn.functional.softmax(logits, dim=0)`
- Get probability of correct class: `probs[target]`
- Loss: `-torch.log(correct_prob)`

**Test case:**
```python
logits = torch.tensor([1.0, 3.0, 0.5])  # Model outputs
target = 1  # True class is index 1

loss = cross_entropy_loss(logits, target)
print(f"Loss: {loss}")
# Should be low because logits[1]=3.0 is highest (correct prediction)
```

**After it works - EXPERIMENT:**
- What's the loss when prediction is perfect (one logit >> others)?
- What's the loss when prediction is wrong (wrong logit is highest)?
- What if all logits are equal? (model is uncertain)
- Compare with PyTorch's built-in: `F.cross_entropy(logits.unsqueeze(0), torch.tensor([target]))`

---

In [13]:
from torch.nn.functional import softmax

def cross_entropy_loss(logits, target):
    prob = softmax(logits, dim=0)
    correct_prob = prob[target]
    return -torch.log(correct_prob)

logits = torch.tensor([1.0, 3.0, 0.5])  # Model outputs
target = 1  # True class is index 1

loss = cross_entropy_loss(logits, target)
print(f"Loss: {loss}")
print(F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])))
# Should be low because logits[1]=3.0 is highest (correct prediction)

Loss: 0.19673413038253784
tensor(0.1967)


In [14]:
logits = torch.tensor([0.0, 1.0, 0.0])  # Model outputs
target = 1  # True class is index 1

loss = cross_entropy_loss(logits, target)
print(f"Loss: {loss}")
print(F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])))

Loss: 0.5514446496963501
tensor(0.5514)


In [15]:
logits = torch.tensor([1.0, 1.0, 1.0])  # Model outputs
target = 1  # True class is index 1

loss = cross_entropy_loss(logits, target)
print(f"Loss: {loss}")
print(F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])))

Loss: 1.0986123085021973
tensor(1.0986)


In [16]:
logits = torch.tensor([3.0, -1.0, 5.0])  # Model outputs
target = 1  # True class is index 1

loss = cross_entropy_loss(logits, target)
print(f"Loss: {loss}")
print(F.cross_entropy(logits.unsqueeze(0), torch.tensor([target])))

Loss: 6.129108905792236
tensor(6.1291)


## **TODO 5: COMPLETE FORWARD PASS WITH YOUR AUDIO**

**Goal:** Run your network on real audio data and compute loss

**Requirements:**
- Load one audio mixture from your dataset
- Extract simple features (mean, std, min, max, energy)
- Pass through your 2-layer network
- Compute loss against true speaker count

**Your task:** Put it all together

**Hints:**
- Simple features: `torch.tensor([audio.mean(), audio.std(), audio.min(), audio.max(), (audio**2).mean()])`
- True label: Get num_speakers from metadata - 1 (to make it 0-indexed)
- Use functions from TODO 2, 3, 4

**Test template:**
```python
# Load one sample (you already know how to do this!)
# ... your data loading code ...

# Extract features
features = ???  # Shape: (5,)

# Initialize random weights
W1 = torch.randn(32, 5)  # 5 inputs → 32 hidden neurons
b1 = torch.randn(32)
W2 = torch.randn(3, 32)  # 32 hidden → 3 outputs
b2 = torch.randn(3)

# Forward pass
output = two_layer_network(features, W1, b1, W2, b2)

# Compute loss
true_label = ???  # 0, 1, or 2 (for 1, 2, or 3 speakers)
loss = cross_entropy_loss(output, true_label)

print(f"Features: {features}")
print(f"Output: {output}")
print(f"True label: {true_label}")
print(f"Loss: {loss}")
```

---

In [17]:
import json
import numpy as np
manifest_path = Path.cwd().parent.parent / "data" / "processed" / "train" / "train_manifest.json"

# Helper function (already provided)
def pad_or_trim_audio(audio_tensor: torch.Tensor, target_length: int) -> torch.Tensor:
    """Pad with zeros or trim audio to target length."""
    current_length = audio_tensor.shape[0]
    
    if current_length < target_length:
        padding = target_length - current_length
        audio_tensor = torch.nn.functional.pad(audio_tensor, (0, padding))
    elif current_length > target_length:
        audio_tensor = audio_tensor[:target_length]
    
    return audio_tensor


def load_audio_with_labels(
    manifest_path: Path,
    num_samples: int = 5,
    target_length: int = 48000,  # 3 seconds at 16kHz
    max_speakers: int = 3,
    device: str = 'cpu'
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, List[Dict]]:
    """
    Load audio mixtures, clean sources, and diarization labels from manifest.
    
    Args:
        manifest_path: Path to train_manifest.json
        num_samples: How many samples to load
        target_length: Target length in samples (pad/trim to this)
        max_speakers: Maximum number of speakers (for padding speaker dimension)
        device: Device to move tensors to
    
    Returns:
        mixtures: (batch, 1, target_length) - Mixed audio
        sources: (batch, max_speakers, target_length) - Clean sources per speaker
        labels: (batch, max_speakers, num_frames) - Diarization labels
        metadata: List of dict with original info
    
    YOUR TASK:
    Load your generated dataset and prepare it for training!
    """
    
    # TODO 1: Load manifest JSON
    # Hint: json.load()
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    # Take only num_samples entries
    manifest = manifest[:num_samples]
    
    # Initialize lists
    mixture_tensors = []
    source_tensors = []
    label_tensors = []
    metadata = []
    
    # TODO 2: Loop through each entry in manifest
    for entry in manifest:
        # TODO 2.1: Load mixture audio
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, sr = load_audio(mixture_path)  # Use load_audio
        
        # TODO 2.2: Convert to tensor and pad/trim
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = pad_or_trim_audio(mixture_tensor, target_length)
        mixture_tensor = mixture_tensor.view(1, 1, -1)  # (1, 1, samples)
        mixture_tensors.append(mixture_tensor)
        
        # TODO 2.3: Load clean sources for each speaker
        speaker_sources = []
        for source_path in entry['source_paths']:
            source_audio, _ = load_audio(source_path)
            source_tensor = torch.from_numpy(source_audio)
            source_tensor = pad_or_trim_audio(source_tensor, target_length)
            speaker_sources.append(source_tensor)
        
        # TODO 2.4: Pad sources to max_speakers if needed
        # If mixture has 2 speakers but max_speakers=3, add one zero tensor
        while len(speaker_sources) < max_speakers:
            speaker_sources.append(torch.zeros(target_length))
        
        # Stack sources: (max_speakers, samples)
        source_tensor = torch.stack(speaker_sources)
        source_tensors.append(source_tensor)
        
        # TODO 2.5: Load diarization labels
        label_path = Path(entry['label_path'])
        labels = np.load(label_path)
        
        # TODO 2.6: Convert labels to tensor
        # Shape is (num_frames, num_speakers)
        # We need (max_speakers, num_frames) with padding
        labels_tensor = torch.from_numpy(labels)
        
        target_frames = 3000
        labels_tensor = labels_tensor.transpose(0, 1)
        current_speakers = labels_tensor.shape[0]
        current_frames = labels_tensor.shape[1]
        
        pad_s = max_speakers - current_speakers
        pad_f = target_frames - current_frames
        
        labels_tensor = torch.nn.functional.pad(
            labels_tensor,
            (0, pad_f, 0, pad_s)
        )
        
        label_tensors.append(labels_tensor)
        
        # Store metadata
        metadata.append({
            'mixture_id': entry['mixture_id'],
            'duration': entry['duration'],
            'num_speakers': entry['num_speakers'],
            'num_utterances': entry['num_utterances'],
            'overlap_ratio': entry['actual_overlap_ratio']
        })
    
    # TODO 3: Stack all tensors into batches
    mixtures_batch = torch.cat(tensors=mixture_tensors, dim=0)  # torch.cat along dim=0
    sources_batch = torch.stack(source_tensors)   # torch.stack
    labels_batch = torch.stack(label_tensors)    # torch.stack
    
    # TODO 4: Move to device
    mixtures_batch = mixtures_batch
    sources_batch = sources_batch
    labels_batch = labels_batch
    
    return mixtures_batch, sources_batch, labels_batch, metadata


In [18]:
manifest_path = Path.cwd().parent.parent / "data" / "processed" / "train" / "train_manifest.json"
print(manifest_path)

print("="*60)
print("LOADING AUDIO BATCH WITH LABELS")
print("="*60)

# Load batch
mixtures, sources, labels, metadata = load_audio_with_labels(
    manifest_path=manifest_path,
    num_samples=1,
    target_length=48000,  # 3 seconds
    max_speakers=3,
    device=device
)

/Users/apple/Documents/programing/projects/voice_isolation/data/processed/train/train_manifest.json
LOADING AUDIO BATCH WITH LABELS


In [19]:
def feature_extraction(
        audio: torch.Tensor
) -> torch.Tensor:
    mean = torch.mean(audio)
    std = torch.std(audio)
    min = torch.min(audio)
    max = torch.max(audio)
    square = audio ** 2
    energy = torch.mean(square)
    
    return torch.stack([mean, std, min, max, energy])

In [20]:
features = feature_extraction(mixtures[0])
W1 = torch.randn(32, 5)
b1 = torch.randn(32)
W2 = torch.randn(3, 32)   # Output layer: 3 classes
b2 = torch.randn(3)

output = two_layer_network(features, W1, b1, W2, b2)

true_label = metadata[0]['num_speakers'] - 1  
loss = cross_entropy_loss(output, true_label)

print(f"Features: {features}")
print(f"Output: {output}")
print(f"True label: {true_label}")
print(f"Loss: {loss}")

Features: tensor([-4.2125e-04,  1.1415e-01, -7.7664e-01,  7.0776e-01,  1.3030e-02])
Output: tensor([-7.1882,  0.0657, -7.1047])
True label: 2
Loss: 7.171847820281982
